In [36]:
import pandas as pd
import numpy as np

In [37]:
df_crm = pd.read_csv('../data/raw/crm_subscribers.csv')

In [38]:
print(f"عدد الاسطر قبل الحذف: {len(df_crm)}")

عدد الاسطر قبل الحذف: 1523


In [39]:
df_crm = df_crm.drop_duplicates(subset=["subscriber_id"]).copy()

In [40]:
print(f"عدد الاسطر بعد الحذف التكرار: {len(df_crm)}")

عدد الاسطر بعد الحذف التكرار: 1500


In [41]:
df_crm["landline_number"] = df_crm["landline_number"].astype(str).str.zfill(9)

df_crm[["subscriber_id", "landline_number"]].head()

,subscriber_id,landline_number
0,SUB-10566,041999001
1,SUB-11101,031920840
2,SUB-11475,021531081
3,SUB-11178,041115559
4,SUB-11009,045811226


In [42]:
condition = [
    df_crm["wilaya"].str.contains("alg", case=False, na=False),
    df_crm["wilaya"].str.contains("oran", case=False, na=False),
    df_crm["wilaya"].str.contains("const", case=False, na=False),
    df_crm["wilaya"].str.contains("masc", case=False, na=False),
]
choices = ["Algeriers", "Oran", "Constantine", "Mascara"]

df_crm["wilaya"] = np.select(condition, choices, default="Unknown")

In [43]:
df_crm["wilaya"].value_counts()

wilaya
Algeriers      549
Oran           509
Constantine    263
Mascara        179
Name: count, dtype: int64

In [44]:
df_crm["speed_mbps"] = pd.to_numeric(
df_crm["speed_mbps"].astype(str).str.replace("Mbps", "", regex=False),
errors="coerce"
)

In [45]:
df_crm[["subscriber_id", "speed_mbps"]].head()

,subscriber_id,speed_mbps
0,SUB-10566,20.0
1,SUB-11101,50.0
2,SUB-11475,15.0
3,SUB-11178,100.0
4,SUB-11009,15.0


In [46]:
plan_price_to_speed = {
    ("Idoom_ADSL", 2000): 15,
    ("Idoom_ADSL", 2200): 20,
    ("Idoom_Fibre", 2000): 60,
    ("Idoom_Fibre", 2200): 120,
    ("Idoom_Fibre", 2600): 180,
    ("Idoom_Fibre", 3000): 240,
    ("Idoom_Fibre", 4200): 300,
}

df_crm["speed_mbps"] = [
    plan_price_to_speed.get((plan, price))
    for plan, price in zip(df_crm["plan_type"], df_crm["monthly_price_dzd"])
]


In [47]:
print("القيم الفارغة في عمود السرعة (speed_mbps):", df_crm["speed_mbps"].isnull().sum())
print(df_crm["speed_mbps"].value_counts())

القيم الفارغة في عمود السرعة (speed_mbps): 0
speed_mbps
120    342
60     335
180    305
240    192
15     162
300     87
20      77
Name: count, dtype: int64


In [48]:
pd.crosstab(
    index=df_crm["plan_type"],
    columns=[df_crm["monthly_price_dzd"], df_crm["speed_mbps"]]
)


monthly_price_dzd 2000      2200      2600 3000 4200
speed_mbps         15   60   20   120  180  240  300
plan_type                                           
Idoom_ADSL         162    0   77    0    0    0    0
Idoom_Fibre          0  335    0  342  305  192   87

In [49]:
df_rech = pd.read_csv('../data/raw/internet_recharges.csv')

In [50]:
df_rech["timestamp"] = pd.to_datetime(df_rech["timestamp"])

In [51]:
failed_mask = df_rech["tx_status"] != "SUCCESS"
df_rech.loc[failed_mask, "amount_dzd"] = 0

In [52]:
pd.crosstab(
    df_rech["tx_status"],
    df_rech["amount_dzd"],
)

amount_dzd,0,2000,2200,2600,3000,4200
tx_status,,,,,,
FAILED,364,0,0,0,0,0
SUCCESS,0,5112,4268,3189,2027,908
TIMEOUT,342,0,0,0,0,0


In [62]:
df_tkt = pd.read_csv('../data/raw/technical_tickets.csv')

In [63]:
df_tkt["opened_at"] = pd.to_datetime(df_tkt["opened_at"])
df_tkt["closed_at"] = pd.to_datetime(df_tkt["closed_at"])

In [64]:
inverted_mask = df_tkt["closed_at"] < df_tkt["opened_at"]
df_tkt.loc[inverted_mask, "closed_at"] = pd.NaT

In [65]:
print("عدد التواريخ المقلوبة المتبقية:", (df_tkt["closed_at"] < df_tkt["opened_at"]).sum())
print("إجمالي القيم الفارغة في closed_at (96 أصلية + 51 مصححة):", df_tkt["closed_at"].isnull().sum())

عدد التواريخ المقلوبة المتبقية: 0
إجمالي القيم الفارغة في closed_at (96 أصلية + 51 مصححة): 147
